In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

In [0]:
df.limit(5).display()

In [0]:
COL_RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in COL_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn("customer_number", F.regexp_replace(col("customer_number"), "-", ""))

In [0]:
df.select("country").distinct().show()

In [0]:
df = df.withColumn(
    "country",
    F.when(col("country") == "DE", "Germany")
     .when(col("country").isin("US", "USA"), "United States")
     .when((col("country") == "") | col("country").isNull(), "unknown")
     .otherwise(col("country"))
)

In [0]:
df.limit(5).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT(5)